# `process_parcels_HIT2d_timeavg2.m` — Python/Jupyter port
This notebook ports the trajectory sampling, per-time pair statistics, time averaging, MAT output, L-curve regularized inversion, and plots. The implementation is in `gridblock_jupyter.py`, which must remain beside this notebook.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from gridblock_jupyter import run_hit_timeavg2, fit_sf3_lcurve

cfg = dict(
 input_dir=Path('/meddy/simingzhang/Data/Parcels_data/HIT2d_rough'),
 nparticles=62500, num_to_select=30000, seconds=20.0, dt=0.1,
 timerange_matlab=np.arange(50,151), random_seed=None,
 save_chunks=False,  # True writes the 202 intermediate chunk MAT files used by MATLAB
 xscale=np.r_[np.arange(2,19),np.arange(21,49,3),np.arange(54,121,6),
             np.arange(132,241,12),np.arange(264,505,24)])
)
cfg

## 1. Pair calculation and time average
With 30,000 particles this evaluates about 45 billion particle pairs across 101 times, just like the MATLAB code, and can be very expensive. `save_chunks=False` avoids unnecessary intermediate disk I/O without changing the final `timeavg5.mat` and `timeavg6.mat` values.

In [ ]:
result, output_paths = run_hit_timeavg2(cfg)

## 2. L-curve spectral-flux inversion
The MATLAB script clears the new result and loads the older `HIT2d_pars_P62500T150timeavg4.mat` before fitting. Set `use_original_timeavg4=True` to reproduce that literal behavior; set it to False to fit the result calculated above.

In [ ]:
use_original_timeavg4 = True
if use_original_timeavg4:
    old = loadmat(cfg['input_dir']/'HIT2d_pars_P62500T150timeavg4.mat', squeeze_me=True)
    SF3_for_fit = np.asarray(old['SF3_mean']).ravel()
    distance_for_fit = np.asarray(old['dist_axis']).ravel()
else:
    SF3_for_fit = result['SF3_mean']
    distance_for_fit = result['dist_axis']

lambda_vec=np.array([1000,100,10,1,1e-1,1e-2])
SpecFlux,Vt,ebs,kf,lf,lambda_opt = fit_sf3_lcurve(
    SF3_for_fit, distance_for_fit, 0.05, 5.0, lambda_vec, plot=True)
print('lambda_opt =',lambda_opt)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].semilogx(distance_for_fit,SF3_for_fit/distance_for_fit,label='SF3/r')
ax[0].semilogx(lf,Vt[:,0]/lf,label='reconstruction'); ax[0].legend(); ax[0].set_xlabel('r')
ax[1].semilogx(2*np.pi/kf,SpecFlux[:,0]); ax[1].set_xlabel('2pi/k'); ax[1].set_ylabel('SpecFlux')
plt.tight_layout()

## Consistency notes
MATLAB does not seed `randperm` in this script, and NumPy uses a different random generator. For exact selected-particle matching, provide MATLAB's `random_indices` rather than relying on a seed. SciPy writes standard MAT files rather than MATLAB `-v7.3`; variables and numerical contents are retained, but the on-disk container format differs. The original `SF2lt` is calculated but never saved or used, so it is not retained in final outputs.